# Stabilité numérique — section 5.9

Ce notebook reproduit les trois tests de stabilité numérique locale du **tableau 8** : nombre de pas $N$, coefficient de relaxation $\lambda$ et tolérance $\varepsilon$.

Le scénario économique est maintenu fixe. Les équations, métriques et conventions proviennent directement du noyau commun `mfg_liquidation`.


In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
from IPython.display import display


def find_project_root(start=None):
    """Locate the repository root from a notebook running inside the project."""
    start = Path.cwd().resolve() if start is None else Path(start).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").is_file() and (candidate / "src" / "mfg_liquidation").is_dir():
            return candidate
    raise RuntimeError(
        "Repository root not found. Start Jupyter from inside the cloned project."
    )


ROOT = find_project_root()
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from mfg_liquidation.metrics import compute_execution_metrics
from mfg_liquidation.reporting import format_decimal_fr, format_integer_fr
from mfg_liquidation.solver import (
    compute_aggregated_flow,
    solve_multi_population_fixed_point,
)

OUTPUT_DIR = ROOT / "results" / "tables"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Projet : {ROOT}")
print(f"Dossier de sortie : {OUTPUT_DIR.resolve()}")


Projet : C:\Users\casol\Desktop\perso\liquidation_MFG
Dossier de sortie : C:\Users\casol\Desktop\perso\liquidation_MFG\results\tables


## Paramètres

Scénario du tableau 8 : $\bar q_0^1=200\,000$, $\bar q_0^2=50\,000$, $\gamma_1=5\times10^{-5}$, $\gamma_2=5\times10^{-7}$, $A_1=5\times10^{-6}$, $A_2=10^{-9}$, $\rho_1=0{,}75$, $\rho_2=0{,}25$.


In [2]:
T = 1.0
LIQUIDITY = 2_000_000.0
ETA = 0.1
SIGMA = 0.3
ALPHA = 8e-8

QBAR0_1, QBAR0_2 = 200_000.0, 50_000.0
GAMMA_1, GAMMA_2 = 5e-5, 5e-7
A_1, A_2 = 5e-6, 1e-9
RHO_1, RHO_2 = 0.75, 0.25

BASE_N = 2000
BASE_EPSILON = 1e-8
BASE_RELAXATION_LAMBDA = 0.35
MAX_ITER = 100_000

N_VALUES = (1000, 2000, 4000)
RELAXATION_LAMBDA_VALUES = (0.20, 0.35, 0.50)
EPSILON_VALUES = (1e-6, 1e-8, 1e-10)


In [3]:
def run_single_scenario(n_steps, epsilon, relaxation_lambda):
    qbar0_list = [np.array([QBAR0_1]), np.array([QBAR0_2])]
    gamma_list = [GAMMA_1, GAMMA_2]
    rho_list = [RHO_1, RHO_2]

    liquidity = np.array([LIQUIDITY], dtype=float)
    eta_diag = np.array([ETA], dtype=float)
    alpha_diag = np.array([ALPHA], dtype=float)
    mathbb_v = np.diag(liquidity / (4.0 * eta_diag))
    impact_matrix = np.diag(alpha_diag)
    a_terminal_list = [np.diag([A_1]), np.diag([A_2])]
    sigma_matrix = np.array([[SIGMA**2]], dtype=float)

    t_grid, qbar, vbar, info = solve_multi_population_fixed_point(
        qbar0_list,
        gamma_list,
        rho_list,
        mathbb_v,
        impact_matrix,
        a_terminal_list,
        sigma_matrix,
        horizon=T,
        n_steps=n_steps,
        relaxation_lambda=relaxation_lambda,
        epsilon=epsilon,
        max_iter=MAX_ITER,
        require_convergence=False,
    )

    mu = compute_aggregated_flow(vbar, rho_list)
    results = {}

    for i in range(2):
        metrics = compute_execution_metrics(
            t_grid, qbar[i], vbar[i], mu,
            alpha_diag, eta_diag, liquidity, sigma_matrix,
        )
        pop = i + 1
        results[f"q̄{pop}(T)"] = float(qbar[i, -1, 0])
        results[f"Volume {pop}"] = float(metrics["volume"][0])
        results[f"Indice unitaire d'impact permanent {pop}"] = float(metrics["permanent_impact_unit"][0])
        results[f"Coût temporaire unitaire {pop}"] = float(metrics["temporary_cost_unit"][0])
        results[f"Exposition au risque de prix normalisée {pop}"] = float(metrics["risk_normalized"])

    results["Nombre d’itérations"] = int(info.n_iter)
    results["Erreur finale"] = float(info.error)
    results["Convergé"] = bool(info.converged)
    return results


def run_numerical_stability_tests():
    rows_n = []
    for n_steps in N_VALUES:
        row = run_single_scenario(n_steps, BASE_EPSILON, BASE_RELAXATION_LAMBDA)
        row["N"] = n_steps
        rows_n.append(row)

    rows_lambda = []
    for relaxation_lambda in RELAXATION_LAMBDA_VALUES:
        row = run_single_scenario(BASE_N, BASE_EPSILON, relaxation_lambda)
        row["lambda"] = relaxation_lambda
        rows_lambda.append(row)

    rows_epsilon = []
    for epsilon in EPSILON_VALUES:
        row = run_single_scenario(BASE_N, epsilon, BASE_RELAXATION_LAMBDA)
        row["epsilon"] = epsilon
        rows_epsilon.append(row)

    return {
        "N": pd.DataFrame(rows_n).set_index("N"),
        "lambda": pd.DataFrame(rows_lambda).set_index("lambda"),
        "epsilon": pd.DataFrame(rows_epsilon).set_index("epsilon"),
    }


## Calcul

In [4]:
stability_results = run_numerical_stability_tests()
print("Calcul terminé.")


Calcul terminé.


In [5]:
def format_epsilon(value):
    exponent = int(round(np.log10(float(value))))
    superscript = str.maketrans("0123456789-", "⁰¹²³⁴⁵⁶⁷⁸⁹⁻")
    return "10" + str(exponent).translate(superscript)


def format_parameter(table_key, value):
    if table_key == "N":
        return str(int(round(float(value))))
    if table_key == "lambda":
        return f"{float(value):g}".replace(".", ",")
    return format_epsilon(float(value))


DISPLAY_COLUMNS = [
    "q̄_T pop.1", "Volume pop.1", "Prix perm. pop.1", "Coût temp. pop.1", "Risque pop.1",
    "q̄_T pop.2", "Volume pop.2", "Prix perm. pop.2", "Coût temp. pop.2", "Risque pop.2",
    "Nombre d’itération",
]

def build_display_table(results, table_key, parameter_label):
    rows = []
    for parameter_value, row in results[table_key].iterrows():
        rows.append([
            format_parameter(table_key, parameter_value),
            format_integer_fr(row["q̄1(T)"]),
            format_integer_fr(row["Volume 1"]),
            format_decimal_fr(row["Indice unitaire d'impact permanent 1"]),
            format_decimal_fr(row["Coût temporaire unitaire 1"]),
            format_decimal_fr(row["Exposition au risque de prix normalisée 1"]),
            format_integer_fr(row["q̄2(T)"]),
            format_integer_fr(row["Volume 2"]),
            format_decimal_fr(row["Indice unitaire d'impact permanent 2"]),
            format_decimal_fr(row["Coût temporaire unitaire 2"]),
            format_decimal_fr(row["Exposition au risque de prix normalisée 2"]),
            format_integer_fr(row["Nombre d’itérations"]),
        ])
    return pd.DataFrame(rows, columns=[parameter_label] + DISPLAY_COLUMNS)

def build_word_text(results, table_key):
    label = {"N": "N", "lambda": "λ", "epsilon": "ε"}[table_key]
    table = build_display_table(results, table_key, label)
    return "\n".join("\t".join(map(str, row)) for row in table.to_numpy())


## Nombre de pas de discrétisation $N$

In [6]:
table_N = build_display_table(stability_results, "N", "N")
display(table_N.style.hide(axis="index"))


N,q̄_T pop.1,Volume pop.1,Prix perm. pop.1,Coût temp. pop.1,Risque pop.1,q̄_T pop.2,Volume pop.2,Prix perm. pop.2,Coût temp. pop.2,Risque pop.2,Nombre d’itération
1000,21,199 277,"-0,0062","0,0351","0,0709",25 366,24 565,"-0,0071","0,0032","0,3780",79
2000,22,199 628,"-0,0062","0,0351","0,0710",25 370,24 596,"-0,0071","0,0032","0,3781",79
4000,22,199 803,"-0,0062","0,0351","0,0711",25 372,24 611,"-0,0071","0,0032","0,3782",79


## Coefficient de relaxation $\lambda$

In [7]:
table_lambda = build_display_table(stability_results, "lambda", "λ")
display(table_lambda.style.hide(axis="index"))


λ,q̄_T pop.1,Volume pop.1,Prix perm. pop.1,Coût temp. pop.1,Risque pop.1,q̄_T pop.2,Volume pop.2,Prix perm. pop.2,Coût temp. pop.2,Risque pop.2,Nombre d’itération
"0,2",22,199 628,"-0,0062","0,0351","0,0710",25 370,24 596,"-0,0071","0,0032","0,3781",150
"0,35",22,199 628,"-0,0062","0,0351","0,0710",25 370,24 596,"-0,0071","0,0032","0,3781",79
"0,5",22,199 628,"-0,0062","0,0351","0,0710",25 370,24 596,"-0,0071","0,0032","0,3781",50


## Tolérance du point fixe $\varepsilon$

In [8]:
table_epsilon = build_display_table(stability_results, "epsilon", "ε")
display(table_epsilon.style.hide(axis="index"))


ε,q̄_T pop.1,Volume pop.1,Prix perm. pop.1,Coût temp. pop.1,Risque pop.1,q̄_T pop.2,Volume pop.2,Prix perm. pop.2,Coût temp. pop.2,Risque pop.2,Nombre d’itération
10⁻⁶,22,199 628,"-0,0062","0,0351","0,0710",25 370,24 596,"-0,0071","0,0032","0,3781",67
10⁻⁸,22,199 628,"-0,0062","0,0351","0,0710",25 370,24 596,"-0,0071","0,0032","0,3781",79
10⁻¹⁰,22,199 628,"-0,0062","0,0351","0,0710",25 370,24 596,"-0,0071","0,0032","0,3781",100 000


## Sauvegarde CSV

In [9]:
for key, frame in stability_results.items():
    frame.to_csv(OUTPUT_DIR / f"robustness_{key}.csv", encoding="utf-8-sig")
    (OUTPUT_DIR / f"robustness_{key}_word.tsv").write_text(
        build_word_text(stability_results, key), encoding="utf-8"
    )
print("Fichiers sauvegardés dans :", OUTPUT_DIR)


Fichiers sauvegardés dans : C:\Users\casol\Desktop\perso\liquidation_MFG\results\tables
